# 04 · From one leaflet to a whole branch: LLS-SevEst 2.0

The first model measured one leaflet per image. In version 2.0 all the leaflets of a lateral branch are scanned
together (six branches), which raises two new problems:

1. **Segmentation.** The original model marked veins and pale parts of the blade as lesion in some leaflets, so I
   tried many combinations of extra pixel features (sections 1 and 2).
2. **Counting.** To account for defoliation, the tool has to count the leaflets and compare them with the expected
   number, 4 per node (sections 3 and 4).

The scans and the tables in `data/` are not included in the repository and are available on reasonable request.
Put the scans in `data/scans/` as `IMG_20250424_000<n>.tif`. Sections 1 and 2 need the scans; the sweeps of
section 3 run from the saved counts in `data/leaflet_counts_*.csv`.

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from sklearn import preprocessing
from sklearn.cluster import KMeans

SCANS_DIR = Path("../../data/scans")
DATA_DIR = Path("../../data")
SCAN_1 = SCANS_DIR / "IMG_20250424_0001.tif"


def read_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## 1. Segmentation: which pixel features?

The 2024 model described each pixel with 6 features (R, G, B and the three differences between them). For the
branches I wrote one function that computes every feature I tried, and ran K-means (3 groups, standardised features)
on each combination: HSV colour, brightness, gradient, position in the image, distance to the centre, normalised
colour, local binary patterns (LBP), and a texture-and-shape set. **I judged each trial by looking at the
segmentation** (were veins and pale areas marked as lesion?), not with a numerical comparison.

`FEATURE_SETS` lists the trials in the order I ran them, with the severity each one gave on scan 1 and the group it
counted as lesion. The 12-feature set `final_12` gave the cleanest segmentation and is the one in the package.
`texture_morphology_8` was run with a Sobel kernel of 3 for the gradient; `build_features` uses 5, as in every
other trial, so its recomputed severity can differ slightly from the recorded one.

In [ ]:
def build_features(img_rgb):
    """Every per-pixel feature tried in 2.0; each trial uses a subset (see FEATURE_SETS)."""
    blur = cv2.medianBlur(img_rgb, 11)
    rows, cols = blur.shape[:2]
    r, c = np.indices((rows, cols))
    px = blur.astype(int)
    f = pd.DataFrame({"row": r.ravel(), "col": c.ravel(),
                      "red": px[..., 0].ravel(), "green": px[..., 1].ravel(), "blue": px[..., 2].ravel()})
    f["diff_r_g"] = f["red"] - f["green"]
    f["diff_r_b"] = f["red"] - f["blue"]
    f["diff_b_g"] = f["blue"] - f["green"]

    hsv = cv2.cvtColor(blur, cv2.COLOR_RGB2HSV)
    f["hue"], f["saturation"], f["value"] = (hsv[..., k].ravel() for k in range(3))
    f["sat_norm"], f["val_norm"] = f["saturation"] / 255.0, f["value"] / 255.0

    gray = cv2.cvtColor(blur, cv2.COLOR_RGB2GRAY)
    gx, gy = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=5), cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=5)
    f["gradient"] = np.sqrt(gx ** 2 + gy ** 2).ravel()
    f["brightness"] = 0.299 * f["red"] + 0.587 * f["green"] + 0.114 * f["blue"]

    f["rel_row"], f["rel_col"] = f["row"] / rows, f["col"] / cols
    f["dist_to_center"] = np.sqrt((f["row"] - rows / 2) ** 2 + (f["col"] - cols / 2) ** 2)
    total = f[["red", "green", "blue"]].sum(axis=1)
    f["norm_r"], f["norm_g"], f["norm_b"] = f["red"] / total, f["green"] / total, f["blue"] / total
    f["lbp"] = local_binary_pattern(gray, 8 * 3, 3, method="uniform").ravel()

    f["gabor"] = cv2.filter2D(gray, cv2.CV_64F, cv2.getGaborKernel((15, 15), 3, np.pi / 4, 10.0, 0.5)).ravel()
    adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    closed = cv2.morphologyEx(cv2.morphologyEx(adaptive, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=2),
                              cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8), iterations=2)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = cv2.arcLength(contours[0].astype(np.int32), True) if contours else 0.0
    area = cv2.moments(closed)["m00"]
    f["perimeter"], f["area"] = perimeter, area
    f["compactness"] = 4 * np.pi * area / (perimeter ** 2 + 1e-6)
    return f


RGB, DIFFS = ["red", "blue", "green"], ["diff_r_g", "diff_r_b", "diff_b_g"]
BASE_11 = RGB + DIFFS + ["hue", "saturation", "value", "gradient", "brightness"]
NORM = ["norm_r", "norm_g", "norm_b"]

# name: (features, severity on scan 1 in %, group counted as lesion)
FEATURE_SETS = {
    "original_6":              (RGB + DIFFS, 6.87, 1),
    "hsv_11":                  (RGB + ["hue", "saturation", "value"] + DIFFS + ["sat_norm", "val_norm"], 7.55, 1),
    "hsv_only_5":              (["hue", "saturation", "value", "sat_norm", "val_norm"], 74.97, 1),
    "texture_morphology_8":    (["hue", "saturation", "value", "gradient", "gabor", "perimeter", "area", "compactness"], 55.89, 1),
    "position_13":             (["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value", "gradient", "brightness"], None, 2),
    "position_no_gradient_12": (["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value", "brightness"], None, 2),
    "position_no_brightness_12":(["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value", "gradient"], None, 2),
    "position_no_hsv_10":      (["row", "col"] + RGB + DIFFS + ["gradient", "brightness"], None, 2),
    "position_no_diffs_10":    (["row", "col"] + RGB + ["hue", "saturation", "value", "gradient", "brightness"], None, 2),
    "position_rgb_diffs_hsv_11":(["row", "col"] + RGB + DIFFS + ["hue", "saturation", "value"], None, 2),
    "base_11":                 (BASE_11, 7.28, 1),
    "base_11_+_all_extras_18": (BASE_11 + ["rel_row", "rel_col", "dist_to_center"] + NORM + ["lbp"], 7.55, 1),
    "base_11_+_pos_lbp_15":    (BASE_11 + ["rel_row", "rel_col", "dist_to_center", "lbp"], 7.93, 1),
    "base_11_+_norm_14":       (BASE_11 + NORM, 57.59, 1),
    "base_11_+_dist_norm_lbp_16": (BASE_11 + ["dist_to_center"] + NORM + ["lbp"], 7.52, 1),
    "base_11_+_dist_norm_15":  (BASE_11 + ["dist_to_center"] + NORM, 57.59, 1),
    "no_hsv_13":               (RGB + DIFFS + ["gradient", "brightness", "dist_to_center"] + NORM + ["lbp"], 57.14, 1),
    "final_12":                (RGB + DIFFS + ["gradient", "dist_to_center"] + NORM + ["lbp"], 7.85, 2),
}


def order_clusters(labels, centers):
    """Renumber the groups by increasing sum of their centre coordinates."""
    order = np.argsort(centers.sum(axis=1))
    remap = {old: new for new, old in enumerate(order)}
    return np.vectorize(remap.get)(labels)


def segment(path, features, random_state=2206):
    """K-means (3 groups, standardised features). Returns the ordered label image and the RGB image."""
    img = read_rgb(path)
    table = build_features(img)
    X = preprocessing.StandardScaler().fit_transform(table[features].to_numpy(dtype=float))
    km = KMeans(n_clusters=3, random_state=random_state, n_init=10).fit(X)
    return order_clusters(km.labels_, km.cluster_centers_).reshape(img.shape[:2]), img


def describe_clusters(img_rgb, ordered):
    """Mean raw RGB colour and share of pixels of each ordered group."""
    flat, lab = img_rgb.reshape(-1, 3), ordered.ravel()
    return pd.DataFrame({g: {"red": flat[lab == g, 0].mean(), "green": flat[lab == g, 1].mean(),
                             "blue": flat[lab == g, 2].mean(), "share_%": 100 * np.mean(lab == g)}
                         for g in np.unique(lab)}).T.round(1)


def severity_from_groups(ordered, lesion_label=2, healthy_label=0):
    lesion, healthy = np.sum(ordered == lesion_label), np.sum(ordered == healthy_label)
    return 100 * lesion / (lesion + healthy)

In [ ]:
results = {}
for name, (features, recorded, lesion_label) in FEATURE_SETS.items():
    ordered, img = segment(SCAN_1, features)
    results[name] = ordered
    print(f"== {name} ({len(features)} features)  | group counted as lesion: {lesion_label}"
          f"  | recorded: {recorded}"
          f"  | recomputed: {severity_from_groups(ordered, lesion_label):.2f}%")
    print(describe_clusters(img, ordered))

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
for ax, (name, ordered) in zip(axes.ravel(), results.items()):
    ax.imshow(ordered, cmap="Oranges_r"); ax.set_title(name, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. What the trials show, and a catch in the group numbers

Most trials give a severity of about 7–8 % on scan 1, but some jump far above it: `hsv_only_5` (75 %),
`texture_morphology_8` (56 %), and `base_11_+_norm_14`, `base_11_+_dist_norm_15` and `no_hsv_13` (about 57 %).
Part of the texture-and-shape set could never help: perimeter, area and compactness are one value per image,
repeated for every pixel, so after standardising they add nothing.

There is also a catch. K-means numbers its groups arbitrarily, so I order them by the sum of their centre coordinates
and count one of the ordered groups as lesion. **Which group is the lesion depends on the features**: group 1 in most
trials, group 2 in the `position_*` trials and in `final_12` (the third element of each entry of `FEATURE_SETS`). A
jump in severity can therefore come from a wrongly numbered group as well as from a real segmentation failure; the
mean colour of each group, printed above, tells them apart.

The same problem appeared with the 6-feature model: in 2 of the 6 scans the lesion came out as group 2 and the
background as group 1, so the percentage was computed against the background (56.5 % and 58.4 %, against 5.5–6.9 %
in the other four). The final tool keeps the ordering rule and counts group 2 as lesion, but only inside the detected
leaflets, which limits how much a wrongly numbered background group can matter. The ordering itself is not checked
automatically, so `check_group_numbers` prints the mean colour of the three groups on each of the six scans.

In [ ]:
def check_group_numbers(scans=range(1, 7), features=FEATURE_SETS["final_12"][0]):
    for n in scans:
        ordered, img = segment(SCANS_DIR / f"IMG_20250424_000{n}.tif", features)
        print(f"scan {n}: severity with group 2 = lesion, group 0 = healthy: {severity_from_groups(ordered):.2f}%")
        print(describe_clusters(img, ordered), "\n")


check_group_numbers()

## 3. Counting leaflets

The counter finds the leaflets as blobs: grey image, Gaussian blur, Otsu threshold (inverted), morphological
opening/closing, contours, then filters by area and by brightness. The expected number is 4 leaflets per node; the
difference is the missing leaflets, and `defoliation = missing / expected`.

The versions I tried, in order. In the first test **3 of the 6 scans counted 2 leaflets too many**, because two
black borders in the upper and lower right corners of the scans, an artefact of the scanner, were counted as
leaflets:

| Version | Blur | Morphology | Filters |
|---|---|---|---|
| `v1` | Gaussian 7 × 7 | ellipse 5 × 5, 2 iterations open and close | area ≥ 100 |
| `v2` | same | same | + discard contours touching the image border |
| `v3` | Gaussian 3 × 3 | ellipse 5 × 5, 1 iteration | area ≥ 100, + discard dark blobs (mean grey < 100) |
| `v4` | Gaussian 5 × 5 | ellipse 5 × 5, 1 iteration | area ≥ 800, optional maximum area, dark threshold 100 |
| `v5` | Gaussian 5 × 5 | ellipse 3 × 3, 1 iteration | same as `v4` |
| **final** | **Gaussian 3 × 3, SD 1.5** | **ellipse 3 × 3, 1 iteration open + 1 close** | area ≥ 800, dark threshold 100 |

Then I ran two systematic sweeps over the six scans, ten configurations each, against the real number of
leaflets of each scan, counted by hand.

In [ ]:
def count_leaflets(path, blur=(3, 3), sigma=1.5, element=3, ops=(("open", 1), ("close", 1)),
                   min_area=800, max_area=None, dark_threshold=100, drop_border_touching=False, return_image=False):
    """Leaflet counter; the defaults are the final parameters, used by the package."""
    bgr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)[:, :, :3]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    if blur:
        gray = cv2.GaussianBlur(gray, blur, sigma)
    mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (element, element))
    operations = {"open": lambda m, n: cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel, iterations=n),
                  "close": lambda m, n: cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel, iterations=n),
                  "dilate": lambda m, n: cv2.dilate(m, kernel, iterations=n),
                  "erode": lambda m, n: cv2.erode(m, kernel, iterations=n)}
    for name, iterations in ops:
        mask = operations[name](mask, iterations)

    height, width = mask.shape
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    leaflets = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area or (max_area is not None and area > max_area):
            continue
        if drop_border_touching:
            x, y, w, h = cv2.boundingRect(c)
            if x <= 1 or y <= 1 or x + w >= width - 1 or y + h >= height - 1:
                continue
        if dark_threshold:
            blob = np.zeros_like(mask)
            cv2.drawContours(blob, [c], -1, 255, thickness=cv2.FILLED)
            if cv2.mean(gray, mask=blob)[0] < dark_threshold:
                continue
        leaflets.append(c)
    if return_image:
        out = bgr.copy(); cv2.drawContours(out, leaflets, -1, (0, 255, 0), 2)
        return len(leaflets), cv2.cvtColor(out, cv2.COLOR_BGR2RGB)
    return len(leaflets)


COUNTER_VERSIONS = {
    "v1":    dict(blur=(7, 7), sigma=0, element=5, ops=(("open", 2), ("close", 2)), min_area=100, dark_threshold=0),
    "v2":    dict(blur=(7, 7), sigma=0, element=5, ops=(("open", 2), ("close", 2)), min_area=100, dark_threshold=0,
                  drop_border_touching=True),
    "v3":    dict(blur=(3, 3), sigma=0, element=5, ops=(("open", 1), ("close", 1)), min_area=100, dark_threshold=100),
    "v4":    dict(blur=(5, 5), sigma=0, element=5, ops=(("open", 1), ("close", 1)), min_area=800, dark_threshold=100),
    "v5":    dict(blur=(5, 5), sigma=0, element=3, ops=(("open", 1), ("close", 1)), min_area=800, dark_threshold=100),
    "final": dict(),
}


def run_sweep(configs, scans=range(1, 7)):
    """Count the leaflets of every scan under every configuration (needs the scans)."""
    return pd.DataFrame({name: {f"IMG_20250424_000{n}.tif": count_leaflets(SCANS_DIR / f"IMG_20250424_000{n}.tif", **kw)
                                for n in scans} for name, kw in configs.items()})

### Gaussian blur and morphology sweeps

These two tables are the saved counts of the sweeps (`data/leaflet_counts_gaussian_sweep.csv` and
`data/leaflet_counts_morphology_sweep.csv`), so this part runs without the scans. Each column is one configuration; the table of errors is
`counted − real`.

* Gaussian sweep: kernels 3 × 3, 5 × 5 and 7 × 7 without SD, with SD 1.5 and with SD 2, and without blur.
* Morphology sweep (with the best blur, 3 × 3 and SD 1.5): the kernel and the iterations of open and close,
  and variants with only open, only close, or an extra dilate or erode.

In [ ]:
gaussian = pd.read_csv(DATA_DIR / "leaflet_counts_gaussian_sweep.csv").set_index("scan")
morphology = pd.read_csv(DATA_DIR / "leaflet_counts_morphology_sweep.csv").set_index("scan")


def errors(table):
    real = table["real_leaflets"]
    return table.drop(columns="real_leaflets").sub(real, axis=0)


for title, table in [("Gaussian blur sweep", gaussian), ("Morphology sweep", morphology)]:
    err = errors(table)
    print(f"== {title}: counted - real, per scan")
    print(err.T.assign(total_abs_error=err.abs().sum().values, exact_scans=(err == 0).sum().values)
             .sort_values(["total_abs_error"]))
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, (title, table) in zip(axes, [("Gaussian blur sweep", gaussian), ("Morphology sweep", morphology)]):
    x = np.arange(len(table))
    for col in table.columns.drop("real_leaflets"):
        ax.plot(x, table[col], "o", alpha=0.5, label=col)
    ax.plot(x, table["real_leaflets"], "k_", markersize=22, markeredgewidth=3, label="real")
    ax.set_xticks(x, [f"scan {i + 1}" for i in x]); ax.set_ylabel("Leaflets"); ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.show()

**What the sweeps say.** The best result is the real count in 5 of the 6 scans, with scan 1 always off by one
(38 counted, 39 real). It comes from a Gaussian blur of 3 × 3 with SD 1.5 (SD 2 gives the same) combined with a
3 × 3 ellipse and one iteration of opening and one of closing, which are the parameters of the final tool. Five other
morphology variants give the same counts: opening once only, closing once only, opening twice, and opening plus
closing with an extra dilation or erosion.

With six scans, "best" means "fewest counting errors on the six scans I used to choose": there is no held-out set.

## 4. Defoliation: how to combine it with the severity

Once the leaflets are counted, the missing ones give `defoliation = missing / expected` (expected = 4 per node;
the scans had 10, 10, 9, 11, 12 and 12 nodes). I compared five ways of combining it with the lesion severity and
kept the multiplicative one:

| Combination | Formula |
|---|---|
| simple average | (severity + defoliation) / 2 |
| weighted | 0.7 × severity + 0.3 × defoliation |
| maximum | max(severity, defoliation) |
| capped sum | min(1, severity + defoliation) |
| **multiplicative (kept)** | **1 − (1 − severity) × (1 − defoliation)** |

The multiplicative form reads as "the proportion of the expected leaf area that is lesion or missing", and cannot
go above 100 %.

In [ ]:
NODES = {1: 10, 2: 10, 3: 9, 4: 11, 5: 12, 6: 12}


def defoliation_variants(severity, defoliation):
    """The five ways of combining severity and defoliation that I compared (all fractions in 0-1)."""
    return {"average": (severity + defoliation) / 2,
            "weighted_0.7_0.3": 0.7 * severity + 0.3 * defoliation,
            "maximum": max(severity, defoliation),
            "capped_sum": min(1, severity + defoliation),
            "multiplicative": 1 - (1 - severity) * (1 - defoliation)}

## 5. Putting it together

The final tool, `lls_sev_est/lls_sev_est.py`, does the whole thing (see the main README for how to use it): it
detects and counts the leaflets, computes defoliation, shaves 7 px off the edge of each leaflet so that the shadow on
the border is not counted as lesion, segments with the 12 features, computes the severity only inside the leaflets,
and combines it with defoliation. The counter parameters and the 12 features in the package are the ones in this
notebook; the 7 px edge protection was set in the package and is not tested here.